<a href="https://colab.research.google.com/github/koziychukal17-byte/A-B-Test-Analysis-Conversion-Metrics-Statistical-Significance/blob/main/A_B_Test_Analysis_Conversion_Metrics_%26_Statistical_Significance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade google-cloud-bigquery

In [ ]:
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy.stats import pearsonr
from scipy.stats import spearmanr
from scipy.stats import normaltest, shapiro
from scipy.stats import kruskal
import statsmodels.api as sm
import numpy as np
import matplotlib.cm as cm # for get_cmap
from matplotlib.colors import to_rgb # Import to_rgb for dynamic text color
from scipy.stats import zscore

In [ ]:
#authentication
auth.authenticate_user()

In [ ]:
client = bigquery.Client(project="data-analytics-mate")

In [ ]:
# SQL-request
query = """
with session_info as (
  SELECT
      s.date,
      s.ga_session_id,
      sp.country,
      sp.device,
      sp.continent,
      sp.channel,
      ab.test,
      ab.test_group
  FROM `DA.ab_test` ab
  join `DA.session` s
  on ab.ga_session_id = s.ga_session_id
  join `DA.session_params` sp
  on sp.ga_session_id = ab.ga_session_id
),

session_with_orders as (
SELECT
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    count(distinct o.ga_session_id) as session_with_orders
FROM `DA.order` o
join session_info
on o.ga_session_id = session_info.ga_session_id
group by
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group
),

events as(
SELECT
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    ep.event_name,
    count(ep.ga_session_id) as event_cnt
FROM `DA.event_params` ep
join session_info
on ep.ga_session_id = session_info.ga_session_id
GROUP BY
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    ep.event_name
),


session as(
select
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    count(distinct session_info.ga_session_id) as session_cnt
from session_info
group by
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group
),

account as(
select
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    count(distinct acs.ga_session_id) as new_account_cnt
from `DA.account_session` acs
join session_info
on acs.ga_session_id = session_info.ga_session_id
group by
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group)

SELECT
    session_with_orders.date,
    session_with_orders.country,
    session_with_orders.device,
    session_with_orders.continent,
    session_with_orders.channel,
    session_with_orders.test,
    session_with_orders.test_group,
    'session with orders' as event_name,
    session_with_orders.session_with_orders as value
from session_with_orders
union all
SELECT
    events.date,
    events.country,
    events.device,
    events.continent,
    events.channel,
    events.test,
    events.test_group,
    event_name,
    event_cnt as value
from events
union all
SELECT
    session.date,
    session.country,
    session.device,
    session.continent,
    session.channel,
    session.test,
    session.test_group,
    'session' as event_name,
    session_cnt as value
from session
union all
SELECT
    account.date,
    account.country,
    account.device,
    account.continent,
    account.channel,
    account.test,
    account.test_group,
    'new acount' as event_name,
    new_account_cnt as value
from account

"""

#perform query
query_job = client.query(query)  # Виконання SQL-запиту
results = query_job.result()  # Очікування завершення запиту


In [ ]:
#perform result to DataFrame
df = results.to_dataframe(create_bqstorage_client=False)

In [ ]:
df.head()

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-12-08,Palestine,desktop,Asia,Direct,4,2,new_account,1
1,2020-12-08,Palestine,desktop,Asia,Direct,3,2,new_account,1
2,2020-11-06,Puerto Rico,desktop,Americas,Social Search,2,2,new_account,1
3,2020-11-06,Puerto Rico,desktop,Americas,Social Search,1,1,new_account,1
4,2020-12-08,Croatia,desktop,Europe,Direct,4,2,new_account,1


In [ ]:
min_max_date = df.groupby("test")["date"].agg(["min", "max"]).reset_index()
min_max_date["date_diff"] = (min_max_date["max"] - min_max_date["min"])
min_max_date["date_diff"] = min_max_date["date_diff"].dt.days + 1
min_max_date

,test,min,max,date_diff
0,1,2020-11-01,2020-11-26,26
1,2,2020-11-01,2020-11-29,29
2,3,2020-11-19,2020-12-20,32
3,4,2020-12-05,2021-01-27,54


Тест 1 тривав 25 днів, тест 2 — 28 днів, тест 3 — 31 день, а тест 4 — 53 дні. Тести 1 та 2 проводилися одночасно, так само як і тести 3 та 4, періоди проведення яких частково перетиналися.

У зв’язку з цим виникає питання, чи була дотримана умова щодо взаємного розділення користувачів між тестами. Також важливо оцінити, наскільки за таких умов ми можемо довіряти отриманим результатам та використовувати їх для подальших висновків.

In [ ]:
#calculate number of events for checking data
event_name_table = pd.pivot_table(df, index="event_name", values="value", aggfunc="sum", columns="test").reset_index()

sessions_by_day = pd.DataFrame({
    "test": min_max_date["test"],
    "total_sessions": [event_name_table.loc[11, t] for t in min_max_date["test"]],
    "duration_days": min_max_date["date_diff"],
})

sessions_by_day["sessions_per_day"] = sessions_by_day["total_sessions"] / sessions_by_day["duration_days"]
sessions_by_day

,test,total_sessions,duration_days,sessions_per_day
0,1,90555,26,3482.884615
1,2,100881,29,3478.655172
2,3,140486,32,4390.187500
3,4,210220,54,3892.962963


Тести мали різну тривалість, тому загальну кількість сесій некоректно порівнювати напряму. Для нормалізації показника розглянуто середню кількість сесій на день: тест 1 — 3 483, тест 2 — 3 479, тест 3 — 4 390, тест 4 — 3 892.

Таким чином, менша кількість сесій у тесті 1 пояснюється передусім коротшою тривалістю тесту.


In [ ]:
#rename "new accout" for snake case
df["event_name"] = df["event_name"].replace("new acount", "new_account")

In [ ]:
from statsmodels.stats.proportion import proportions_ztest
import pandas as pd

#define metrics
metrics = [
    ("add_payment_info", "session"),
    ("add_shipping_info", "session"),
    ("begin_checkout", "session"),
    ("new_account", "session")
]

#define segments
segments = ["country", "device", "continent", "channel"]
all_results = [] #create list for noting results

#  a minimum number of successes and failures in each group
min_count = 5

#function for calculation metrics by segments
def calc_metrics_for_group(df_filtered, current_test_id, segment, segment_value):
  rows = []
  for numerator, denominator in metrics:
    num = df_filtered[df_filtered["event_name"] == numerator].groupby("test_group")["value"].sum()
    den = df_filtered[df_filtered["event_name"] == denominator].groupby("test_group")["value"].sum()


    num_control, num_test = num.get(1), num.get(2)
    den_control, den_test = den.get(1), den.get(2)


    row = {
                      "segment": segment,
                      "segment_value": segment_value,
                      "test_number": current_test_id,
                      "metric": f"{numerator} / {denominator}",
                      "numerator_control": num_control,
                      "numerator_test": num_test,
                      "denominator_control": den_control,
                      "denominator_test": den_test,
                      "control_conversion": None,
                      "test_conversion": None,
                      "metrics_change": None,
                      "z-state": None,
                      "p_value": None,
                      "significant": None,
                  }

#create condition for preventing errors
    if all(v is not None for v in [num_control, num_test, den_control, den_test]) and den_control > 0 and den_test > 0:
        row["control_conversion"] = num_control / den_control
        row["test_conversion"] = num_test / den_test

        #check minimum sample size before running the z-test (successes and failures in both groups)
        enough_sample = (
            num_control >= min_count and (den_control - num_control) >= min_count and
            num_test >= min_count and (den_test - num_test) >= min_count
        )

        if enough_sample:
            z_stat, p_value = proportions_ztest([num_test, num_control], [den_test, den_control])
            row["metrics_change"] = (row["test_conversion"] / row["control_conversion"] - 1) * 100
            row["z-state"] = z_stat
            row["p_value"] = p_value
            row["significant"] = p_value < 0.05

    rows.append(row)
  return rows


#by segment: total + breakdown by value
for segment in segments:
    df_segment = (df.groupby([segment, "event_name", "test", "test_group"])["value"].sum().reset_index())

    for current_test_id in df_segment["test"].unique():
        #total within this segment
        df_total = df_segment[df_segment["test"] == current_test_id]
        df_total_grouped = df_total.groupby(["event_name", "test_group"])["value"].sum().reset_index()
        all_results += calc_metrics_for_group(df_total_grouped, current_test_id, segment, "Total")

        #divide by segment value
        for segment_value in df_segment[segment].unique():
            df_filtered = df_segment[(df_segment[segment] == segment_value) & (df_segment["test"] == current_test_id)]
            all_results += calc_metrics_for_group(df_filtered, current_test_id, segment, segment_value)


results_df = pd.DataFrame(all_results)
results_df

,segment,segment_value,test_number,metric,numerator_control,numerator_test,denominator_control,denominator_test,control_conversion,test_conversion,metrics_change,z-state,p_value,significant
0,country,(not set),1,add_payment_info / session,16.0,19.0,369,373,0.043360,0.050938,17.476542,0.486827,0.626381,False
1,country,(not set),1,add_shipping_info / session,23.0,26.0,369,373,0.062331,0.069705,11.831216,0.404423,0.685902,False
2,country,(not set),1,begin_checkout / session,26.0,36.0,369,373,0.070461,0.096515,36.976696,1.282314,0.199733,False
3,country,(not set),1,new_account / session,29.0,28.0,369,373,0.078591,0.075067,-4.483683,-0.180216,0.856983,False
4,country,Albania,1,add_payment_info / session,1.0,NaN,9,16,NaN,NaN,NaN,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1947,channel,Social Search,4,new_account / session,667.0,653.0,7961,8056,0.083783,0.081058,-3.253444,-0.627241,0.530501,False
1948,channel,Undefined,4,add_payment_info / session,496.0,566.0,5716,5862,0.086774,0.096554,11.270787,1.822812,0.068332,False
1949,channel,Undefined,4,add_shipping_info / session,640.0,666.0,5716,5862,0.111966,0.113613,1.470701,0.280026,0.779457,False
1950,channel,Undefined,4,begin_checkout / session,1578.0,1600.0,5716,5862,0.276067,0.272944,-1.131171,-0.376454,0.706579,False


В табриці results_df та results_df_total розраховано такі показники для (тестової та контрольної груп):

* конверсію для тестової та контрольної груп;
* відносна зміна конверсії;
* z-статистику, визначено p-value та статистичну значущість результату (виколистовуючи proportions_ztest).

Розрахунок виконується в розрізі тестів.

В таблиці results_df пораховано результат по сегментам (country, device, continent, channel).

Окремо можна зазначити:

Перед розрахунком z-тесту для кожної групи (контрольної та тестової) перевіряється мінімальна достатня кількість спостережень — за правилом нормального наближення пропорції потрібно щонайменше 5 "успіхів" (подій чисельника) і 5 "невдач" (різниця між знаменником і чисельником) в обох групах одночасно. Якщо ця умова не виконується, z-тест не проводиться, а p-value для такого рядка залишається порожнім — це запобігає розрахунку статистично ненадійних результатів на замалих вибірках.

Частина результатів не має p-value саме через недостатню кількість спостережень у відповідних групах (тобто тест не проводився).

In [ ]:
from numpy.ma.core import concatenate

metrics = [
    ("add_payment_info", "session"),
    ("add_shipping_info", "session"),
    ("begin_checkout", "session"),
    ("new_account", "session")
]

#  a minimum number of successes and failures in each group
min_count = 5

#create function for each test (total)
results_table = []
test_ids = df['test'].unique()
for current_test_id in test_ids:

    df_filtered = df[df['test'] == current_test_id]

    for numerator, denominator in metrics:
        num = df_filtered[df_filtered['event_name'] == numerator].groupby('test_group')['value'].sum()
        den = df_filtered[df_filtered['event_name'] == denominator].groupby('test_group')['value'].sum()



        num_control = num.get(1)
        num_test = num.get(2)

        den_control = den.get(1)
        den_test = den.get(2)

        row = {
            'test_number': current_test_id,
            'metric': f"{numerator} / {denominator}",
            'numerator_control': num_control,
            'numerator_test': num_test,
            'denominator_control': den_control,
            'denominator_test': den_test,
            'control_conversion': None,
            'test_conversion': None,
            'metrics_change': None,
            'z-state': None,
            'p_value': None,
            'significant': None,
        }


        if all(v is not None for v in [num_control, num_test, den_control, den_test]) \
           and den_control > 0 and den_test > 0:

            row['control_conversion'] = num_control / den_control
            row['test_conversion'] = num_test / den_test

            #check minimum sample size before running the z-test (successes and failures in both groups)
            enough_sample = (
                num_control >= min_count and (den_control - num_control) >= min_count and
                num_test >= min_count and (den_test - num_test) >= min_count
            )

            if enough_sample:
                z_stat, p_value = proportions_ztest(
                    [num_test, num_control],
                    [den_test, den_control]
                )
                row["metrics_change"] = (row["test_conversion"] / row["control_conversion"] - 1) * 100
                row["z-state"] = z_stat
                row['p_value'] = p_value
                row['significant'] = p_value < 0.05

        results_table.append(row)

results_df_total = pd.DataFrame(results_table)

#cobine tables by segments and total
finall_result = pd.concat([results_df, results_df_total], axis=0, ignore_index=True)

#fill missing values in "segment" and "segment_value" for tests
finall_result['segment'] = finall_result['segment'].fillna("Total")
finall_result['segment_value'] = finall_result['segment_value'].fillna("Total by test")
#create mask to name totals for segments
mask_total = finall_result["segment_value"] == "Total"
finall_result.loc[mask_total, "segment_value"] = "Total by " + finall_result.loc[mask_total, "segment"]
finall_result

,segment,segment_value,test_number,metric,numerator_control,numerator_test,denominator_control,denominator_test,control_conversion,test_conversion,metrics_change,z-state,p_value,significant
0,country,(not set),1,add_payment_info / session,16.0,19.0,369,373,0.043360,0.050938,17.476542,0.486827,0.626381,False
1,country,(not set),1,add_shipping_info / session,23.0,26.0,369,373,0.062331,0.069705,11.831216,0.404423,0.685902,False
2,country,(not set),1,begin_checkout / session,26.0,36.0,369,373,0.070461,0.096515,36.976696,1.282314,0.199733,False
3,country,(not set),1,new_account / session,29.0,28.0,369,373,0.078591,0.075067,-4.483683,-0.180216,0.856983,False
4,country,Albania,1,add_payment_info / session,1.0,NaN,9,16,NaN,NaN,NaN,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1963,Total,Total by test,2,new_account / session,4165.0,4184.0,50637,50244,0.082252,0.083274,1.241934,0.588793,0.556000,False
1964,Total,Total by test,1,add_payment_info / session,1988.0,2229.0,45362,45193,0.043825,0.049322,12.542021,3.924884,0.000087,True
1965,Total,Total by test,1,add_shipping_info / session,3034.0,3221.0,45362,45193,0.066884,0.071272,6.560481,2.603571,0.009226,True
1966,Total,Total by test,1,begin_checkout / session,3784.0,4021.0,45362,45193,0.083418,0.088974,6.660587,2.978783,0.002894,True


Таблиця finall_result містить об'єднані дані з таблиць  results_df та results_df_total методом "concat" з бібліотеки Pandas.

Результати цієї таблиці були додані в Tableau для створення дашборду. Частина дашборду, де показано статистично значуща зміна конверсії, враховує вибірки, де кількість сесій (метрика session) перевищує 10 000, оскільки менший обсяг вибірки може призводити до менш надійних результатів та випадкових коливань метрик.

In [ ]:
#fill na meaning
finall_result_export = finall_result.fillna({
    "control_conversion": 0,
    "test_conversion": 0,
})

#save "finall_result" in csv
finall_result_export.to_csv("finall_result.csv", index=False)

from google.colab import files
files.download("finall_result.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#**The key takeaways**

##Тест №1

По першому тесту метрики в ціломі показують позитивну статистично значучу зміну конверсії, а саме add_payment_info / session ріст на 12,54, add_shipping_info / session та begin_checkout / session по 6,65.

Та по new_account / session зменшення конверції на 3,35 по тесту, але зменшення не є статистично значучим, тому недостатньо доказів, щоб пов'язувати це з впровадженням зміни. Хоча в розрізі country = United State негативний статистично значуча зміна на 9,72%.

Окремо хочу зазначити, що метрика  add_payment_info / session по Channel = Direct має значний статистично значущий ріст (на 35,83%) та падіня Organic Search на 19,46. Враховуючи загальний позитивний та статистично значущий ріст за трьома ключовими метриками (add_payment_info / session, add_shipping_info / session, begin_checkout / session), ця зміна має великий потенціал.

Однак, варто врахувати значне падіння метрики add_payment_info / session для сегменту Organic Search та new_account / session для сегменту United States. Це може вимагати додаткового дослідження причин цього падіння, або ж впровадження зміни з посиленим моніторингом цих сегментів, аби уникнути можливих негативних наслідків.

##Тест №2

Всі метрики демонструють позитивну динаміку, однак зміна не є статистично значущою. Тому отриманих результатів недостатньо для підтвердження позитивного впливу тестової зміни. Оскільки тест уже завершено і продовжити збір даних ми не можемо, тому впроваджувати зміну не потрібно.

У розрізах були виявлені окремі статистично значущі позитивні зміни, зокрема для United States: add_payment_info / session зросла на 20,24%. Для підтвердження результату рекомендується повторне тестування на відповідному сегменті.

##Тест №3

Зміну, що тестувалась в третьому тесті впроваджувати не потрібно, оскільки по всім метрикам прослідковується зменшення конверсії (окрім add_payment_info / session статистично не значуще збільшення на 1,47%). По метрикам add_shipping_info / session і new_account / session статистично не значуще, тобто немає підтвердження, що зменшення через зміну. Метрика begin_checkout / session зменшення статистично значуще конверція зменшилась на 3,35%.

##Тест №4

Зміну за результатами четвертого тесту не рекомендується впроваджувати, оскільки всі ключові метрики демонструють негативну динаміку. Частина показників має статистично значуще зниження.

Тест проводився на найбільшій вибірці, тому отримані результати є достатньо надійними для прийняття рішення.




* *Додатково рекомендую перевірити чи було розділення користувачів між тестами, оскільки зараз немає достатньо даних для перевірки.*

https://drive.google.com/file/d/1OSIX89TnLCh_W0mka2sQHM6AgYwtJE9U/view?usp=sharing

https://public.tableau.com/app/profile/olena.pokoievych/viz/ABtest2_17836804260380/ABTestAnalysisConversionMetricsStatisticalSignificance